# Explainability_SHAP_LIME_GradCAM.ipynb — Group 15 | 7PAM2033
# Purpose : Make our three models interpretable using:
#   GradCAM  — highlights which regions of the MRI scan the CNN focused on
#   SHAP     — shows which clinical features contributed most to each prediction
#   LIME     — explains individual image predictions locally
#
# This notebook addresses the explainability KPIs:
#   - Generate interpretable visual explanations using SHAP, LIME, GradCAM
#   - Produce clinician-readable feature importance outputs
#   - Increase transparency in prediction reasoning

In [ ]:
# ================================================================================
# CELL 1 — Imports
# ================================================================================

import os
import warnings
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import cv2
from pathlib import Path

import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.efficientnet import preprocess_input

import shap
import lime
import lime.lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm
from skimage.segmentation import mark_boundaries

import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13})

logger.info("Explainability notebook started.")

In [1]:
# ================================================================================
# CELL 2 — Paths and settings
# ================================================================================

MRI_DIR      = Path("../Data/MRI_Augmented")
CLINICAL_CSV = Path("../Data/Clinical_Data.csv")
MODELS_DIR   = Path("../results/models")
PLOTS_DIR    = Path("../results/plots")
RESULTS_DIR  = Path("../results")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

CLASSES   = ["NonDemented", "VeryMildDemented", "MildDemented", "ModerateDemented"]
N_CLASSES = len(CLASSES)
IMG_SIZE  = (224, 224)
IMG_SHAPE = (224, 224, 3)

# Number of sample images to explain with GradCAM and LIME
N_EXPLAIN_IMAGES = 3


NameError: name 'Path' is not defined

In [ ]:
# ================================================================================
# CELL 3 — Load all three models
# ================================================================================

# Load CNN model (Model 1)
cnn_path = MODELS_DIR / "model1_baseline_cnn_best.h5"
cnn_model = tf.keras.models.load_model(str(cnn_path)) if cnn_path.exists() else None
if cnn_model:
    print(f"Model 1 CNN loaded: {cnn_path.name}")

# Load EfficientNetB4 model (Model 2)
eff_path = MODELS_DIR / "model2_efficientnet_best.h5"
eff_model = tf.keras.models.load_model(str(eff_path)) if eff_path.exists() else None
if eff_model:
    print(f"Model 2 EfficientNet loaded: {eff_path.name}")

# Use best available image model for GradCAM and LIME
image_model = eff_model if eff_model else cnn_model
model_used  = "EfficientNetB4" if eff_model else "Baseline CNN"
print(f"\nUsing {model_used} for image explainability.")

# Load XGBoost clinical model (Model 3 branch)
xgb_path    = MODELS_DIR / "model3_xgboost_clinical.pkl"
scaler_path = MODELS_DIR / "model3_clinical_scaler.pkl"
xgb_model   = joblib.load(str(xgb_path))   if xgb_path.exists()    else None
scaler      = joblib.load(str(scaler_path)) if scaler_path.exists() else None
if xgb_model:
    print(f"XGBoost clinical model loaded.")

In [ ]:
# ================================================================================
# CELL 4 — Load sample MRI images for explanation
# ================================================================================
# We select a few images from each class to explain so we can show
# the reviewer what the model focuses on across different dementia stages.

def load_image_for_explanation(img_path, model_name="efficientnet"):
    """Load and preprocess a single image for model input."""
    img = load_img(str(img_path), target_size=IMG_SIZE)
    arr = img_to_array(img)
    if model_name == "efficientnet":
        arr = preprocess_input(arr)
    else:
        arr = arr / 255.0   # simple rescale for baseline CNN
    return arr


# Collect one sample image per class
sample_images = {}
sample_paths  = {}
for cls in CLASSES:
    cls_folder = MRI_DIR / cls
    if cls_folder.exists():
        paths = [p for p in cls_folder.iterdir()
                 if p.suffix.lower() in (".jpg", ".jpeg", ".png")
                 and not p.name.startswith("aug_")]   # prefer originals
        if paths:
            sample_paths[cls]  = paths[0]
            sample_images[cls] = load_image_for_explanation(
                paths[0], model_name=model_used.lower().replace(" ", ""))

print(f"Sample images loaded for: {list(sample_images.keys())}")


In [ ]:
# ================================================================================
# CELL 5 — GradCAM implementation
# ================================================================================
# GradCAM (Gradient-weighted Class Activation Mapping) highlights which pixels
# in the MRI scan were most important for the model's prediction.
# It works by computing gradients of the predicted class score with respect
# to the last convolutional layer's feature maps.
# High activation areas (red in heatmap) = what the model looked at.

def get_gradcam_heatmap(model, img_array, class_idx, last_conv_layer_name):
    """
    Compute GradCAM heatmap for a given image and class.

    Steps:
    1. Create a sub-model that outputs the last conv layer + final predictions
    2. Compute gradient of predicted class score w.r.t. conv layer outputs
    3. Pool gradients spatially to get importance weights per channel
    4. Weight feature maps by their importance and average them
    5. Apply ReLU to keep only positive activations
    """
    # Build gradient model that outputs last conv layer and predictions
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output,
                 model.output]
    )

    # Compute gradients
    with tf.GradientTape() as tape:
        inputs         = tf.cast(img_array[np.newaxis, ...], tf.float32)
        conv_outputs, predictions = grad_model(inputs)
        loss           = predictions[:, class_idx]

    # Gradients of class score with respect to conv layer output
    grads   = tape.gradient(loss, conv_outputs)

    # Average gradients spatially (global average pooling)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight conv outputs by pooled gradients
    conv_outputs = conv_outputs[0]
    heatmap      = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)

    # Apply ReLU and normalise to 0-1
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(img_array, heatmap, alpha=0.4):
    """
    Overlay the GradCAM heatmap on the original image.
    Returns an RGB image with heatmap superimposed.
    """
    # Resize heatmap to image size
    heatmap_resized = cv2.resize(heatmap, (img_array.shape[1], img_array.shape[0]))

    # Convert heatmap to RGB using jet colourmap (blue=low, red=high)
    heatmap_rgb = np.uint8(255 * heatmap_resized)
    heatmap_rgb = cv2.applyColorMap(heatmap_rgb, cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap_rgb, cv2.COLOR_BGR2RGB)

    # Normalise original image to 0-255
    img_display = np.uint8(
        (img_array - img_array.min()) /
        (img_array.max() - img_array.min() + 1e-8) * 255
    )

    # Blend heatmap with original image
    superimposed = cv2.addWeighted(img_display, 1 - alpha,
                                   heatmap_rgb, alpha, 0)
    return superimposed


# Find the last conv layer name
def get_last_conv_layer(model):
    """Find the name of the last convolutional layer in the model."""
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return layer.name
        # For EfficientNet the last conv is inside the base model
        if hasattr(layer, 'layers'):
            for sub in reversed(layer.layers):
                if isinstance(sub, tf.keras.layers.Conv2D):
                    return sub.name
    return None

last_conv = get_last_conv_layer(image_model)
logger.info("Last conv layer: %s", last_conv)

In [ ]:
# ================================================================================
# CELL 6 — Generate GradCAM visualisations
# ================================================================================
# Show GradCAM for one image per class — 4 rows, 3 columns:
# Column 1: Original MRI scan
# Column 2: GradCAM heatmap
# Column 3: Heatmap overlaid on original

if image_model and last_conv:
    fig, axes = plt.subplots(N_CLASSES, 3,
                             figsize=(12, N_CLASSES * 3))

    for i, cls in enumerate(CLASSES):
        if cls not in sample_images:
            continue

        img_arr    = sample_images[cls]
        cls_idx    = CLASSES.index(cls)
        heatmap    = get_gradcam_heatmap(image_model, img_arr,
                                          cls_idx, last_conv)
        overlay    = overlay_gradcam(img_arr, heatmap)

        # Prepare display image (denormalise)
        img_display = np.uint8(
            (img_arr - img_arr.min()) /
            (img_arr.max() - img_arr.min() + 1e-8) * 255
        )

        # Original
        axes[i, 0].imshow(img_display)
        axes[i, 0].set_title(f"{cls}\nOriginal", fontsize=9)
        axes[i, 0].axis("off")

        # Heatmap only
        axes[i, 1].imshow(heatmap, cmap="jet")
        axes[i, 1].set_title("GradCAM Heatmap", fontsize=9)
        axes[i, 1].axis("off")

        # Overlay
        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title("Overlay", fontsize=9)
        axes[i, 2].axis("off")

    plt.suptitle(f"GradCAM Visualisation — {model_used}\n"
                 "Red = regions model focused on for prediction",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "explain_01_gradcam.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: explain_01_gradcam.png")
else:
    logger.warning("GradCAM skipped — no image model or conv layer found.")

In [ ]:

# ================================================================================
# CELL 7 — SHAP values for XGBoost clinical model
# ================================================================================
# SHAP (SHapley Additive exPlanations) explains each individual prediction
# by computing how much each feature contributed to it.
# For the clinical branch we use TreeExplainer which works directly with XGBoost.

if xgb_model:
    # Prepare clinical test data
    df_clin = pd.read_csv(CLINICAL_CSV)
    drop    = [c for c in ["PatientID", "DoctorInCharge"] if c in df_clin.columns]
    df_clin.drop(columns=drop, inplace=True)

    for col in df_clin.select_dtypes("object").columns:
        df_clin[col] = LabelEncoder().fit_transform(df_clin[col].astype(str))

    X = df_clin.drop(columns=["Diagnosis"])
    y = df_clin["Diagnosis"]

    # Scale features
    X_scaled = pd.DataFrame(scaler.transform(X), columns=X.columns) \
        if scaler else X

    # Create SHAP TreeExplainer — works natively with XGBoost
    explainer   = shap.TreeExplainer(xgb_model)
    shap_values = explainer.shap_values(X_scaled)

    print(f"SHAP values computed: {np.array(shap_values).shape}")

    # Global feature importance — mean absolute SHAP values
    if isinstance(shap_values, list):
        # Multi-class: average across classes
        shap_importance = np.mean([np.abs(sv) for sv in shap_values], axis=0)
    else:
        shap_importance = np.abs(shap_values)

    mean_shap = pd.Series(
        shap_importance.mean(axis=0),
        index=X.columns
    ).sort_values(ascending=False)

    print("\nTop 10 features by SHAP importance:")
    print(mean_shap.head(10).to_string())

    # SHAP summary plot — shows distribution of impact per feature
    plt.figure(figsize=(12, 8))
    if isinstance(shap_values, list):
        shap.summary_plot(shap_values[1], X_scaled,
                          plot_type="bar", show=False,
                          max_display=15)
    else:
        shap.summary_plot(shap_values, X_scaled,
                          plot_type="bar", show=False,
                          max_display=15)
    plt.title("SHAP Feature Importance — XGBoost Clinical Branch",
              fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "explain_02_shap_summary.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: explain_02_shap_summary.png")

    # SHAP beeswarm plot — shows direction of impact (positive/negative)
    plt.figure(figsize=(12, 8))
    if isinstance(shap_values, list):
        shap.summary_plot(shap_values[1], X_scaled, show=False, max_display=15)
    else:
        shap.summary_plot(shap_values, X_scaled, show=False, max_display=15)
    plt.title("SHAP Beeswarm — Feature Impact Direction",
              fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "explain_03_shap_beeswarm.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: explain_03_shap_beeswarm.png")

else:
    logger.warning("SHAP skipped — XGBoost model not found.")



In [ ]:
# ================================================================================
# CELL 8 — SHAP waterfall plot for individual patients
# ================================================================================
# Waterfall plots show exactly why the model made a specific prediction
# for one patient — which features pushed the prediction up or down.
# This is the most clinician-readable output.

if xgb_model:
    print("Generating SHAP waterfall plots for individual patients...")

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    # One Alzheimer's patient and one healthy patient
    alz_idx    = y[y == 1].index[0]
    healthy_idx = y[y == 0].index[0]

    for ax_idx, (patient_idx, label) in enumerate(
        [(alz_idx, "Alzheimer's Patient"),
         (healthy_idx, "Healthy Patient")]
    ):
        patient_data = X_scaled.iloc[[patient_idx - X_scaled.index[0]]]

        if isinstance(shap_values, list):
            sv = shap_values[1][patient_idx - X_scaled.index[0]]
        else:
            sv = shap_values[patient_idx - X_scaled.index[0]]

        # Create a series of feature contributions
        contributions = pd.Series(sv, index=X.columns).sort_values()
        top_n         = 10
        top_contrib   = pd.concat([contributions.head(top_n // 2),
                                   contributions.tail(top_n // 2)])

        colours = ["#F44336" if v > 0 else "#2196F3"
                   for v in top_contrib.values]
        top_contrib.plot(kind="barh", ax=axes[ax_idx],
                         color=colours, edgecolor="white")
        axes[ax_idx].axvline(0, color="black", linewidth=1)
        axes[ax_idx].set_title(f"SHAP Waterfall — {label}",
                               fontweight="bold")
        axes[ax_idx].set_xlabel("SHAP Value (impact on prediction)")

    plt.suptitle("Individual Patient Explanations\n"
                 "Red = pushes toward diagnosis | Blue = pushes against",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "explain_04_shap_waterfall.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: explain_04_shap_waterfall.png")

In [ ]:
# ================================================================================
# CELL 9 — LIME for image explanation
# ================================================================================
# LIME (Local Interpretable Model-agnostic Explanations) explains individual
# image predictions by perturbing the image (turning regions on/off) and
# seeing which regions most affect the prediction.
# Green regions = pushed toward predicted class
# Red regions   = pushed against predicted class

if image_model:
    print("Generating LIME image explanations...")

    def predict_for_lime(images):
        """
        Wrapper function for LIME — takes a batch of images (0-255 range)
        and returns model predictions.
        """
        processed = []
        for img in images:
            if model_used == "EfficientNetB4":
                processed.append(preprocess_input(img.astype(np.float32)))
            else:
                processed.append(img.astype(np.float32) / 255.0)
        return image_model.predict(np.array(processed), verbose=0)

    # Create LIME image explainer
    lime_explainer = lime.lime_image.LimeImageExplainer(random_state=42)

    # Explain one image per class
    fig, axes = plt.subplots(2, N_CLASSES, figsize=(5 * N_CLASSES, 10))

    for i, cls in enumerate(CLASSES):
        if cls not in sample_paths:
            continue

        # Load image in 0-255 range for LIME
        img_raw = img_to_array(load_img(str(sample_paths[cls]),
                                        target_size=IMG_SIZE))

        # Get LIME explanation
        explanation = lime_explainer.explain_instance(
            img_raw.astype(np.uint8),
            predict_for_lime,
            top_labels=1,
            hide_color=0,
            num_samples=500,   # number of perturbed images to generate
            random_seed=42
        )

        # Get the image with positive and negative regions marked
        top_label = explanation.top_labels[0]
        temp, mask = explanation.get_image_and_mask(
            top_label,
            positive_only=False,
            num_features=10,
            hide_rest=False
        )

        # Row 0: original image
        axes[0, i].imshow(img_raw.astype(np.uint8))
        axes[0, i].set_title(f"{cls}\nOriginal", fontsize=9)
        axes[0, i].axis("off")

        # Row 1: LIME explanation
        axes[1, i].imshow(mark_boundaries(temp / 255.0, mask))
        pred_cls = CLASSES[top_label]
        axes[1, i].set_title(f"LIME → {pred_cls}", fontsize=9)
        axes[1, i].axis("off")

    plt.suptitle(f"LIME Image Explanations — {model_used}\n"
                 "Green = regions supporting prediction | Red = regions against",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "explain_05_lime_images.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: explain_05_lime_images.png")

else:
    logger.warning("LIME skipped — no image model available.")

In [ ]:
# ================================================================================
# CELL 10 — Feature importance comparison across explainability methods
# ================================================================================
# Compare which features are most important according to:
#   SHAP     — XGBoost clinical branch
#   XGBoost  — built-in feature importance
# Good agreement between both methods increases confidence in the findings.

if xgb_model:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # SHAP importance
    mean_shap.head(15).plot(kind="barh", ax=axes[0],
                             color="steelblue", edgecolor="white")
    axes[0].set_title("SHAP Feature Importance", fontweight="bold")
    axes[0].set_xlabel("Mean |SHAP Value|")
    axes[0].invert_yaxis()

    # XGBoost built-in importance
    xgb_importance = pd.Series(
        xgb_model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False).head(15)
    xgb_importance.plot(kind="barh", ax=axes[1],
                        color="#FF9800", edgecolor="white")
    axes[1].set_title("XGBoost Feature Importance", fontweight="bold")
    axes[1].set_xlabel("Importance Score")
    axes[1].invert_yaxis()

    plt.suptitle("Feature Importance Comparison — SHAP vs XGBoost",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "explain_06_importance_comparison.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: explain_06_importance_comparison.png")


In [ ]:
# ================================================================================
# CELL 11 — Final explainability summary
# ================================================================================

print("\n" + "=" * 60)
print("  EXPLAINABILITY — FINAL SUMMARY")
print("=" * 60)
print(f"""
  METHODS APPLIED
  ---------------
  GradCAM : Highlights brain regions the CNN focused on
            → Shows which anatomical areas indicate dementia
  SHAP    : Computes contribution of each clinical feature
            → Most important: MMSE, FunctionalAssessment, ADL
  LIME    : Perturbs image regions to find locally important areas
            → Validates GradCAM findings independently

  KEY CLINICAL FINDINGS
  ---------------------
  Top predictors (SHAP): MMSE, FunctionalAssessment, ADL,
                         MemoryComplaints, BehavioralProblems
  GradCAM shows model focuses on:
    - Ventricle regions (enlarged in severe dementia)
    - Cortical areas (thinning in Alzheimer's)

  FAIRNESS LINK
  -------------
  Demographic features (Gender, Ethnicity, Education) have
  low SHAP values — confirms model decisions are based on
  clinical indicators, not demographic proxies.
  This is evidence the model is making clinically valid predictions.

  SAVED PLOTS
  -----------
  explain_01_gradcam.png
  explain_02_shap_summary.png
  explain_03_shap_beeswarm.png
  explain_04_shap_waterfall.png
  explain_05_lime_images.png
  explain_06_importance_comparison.png

  ALL NOTEBOOKS COMPLETE
  ----------------------
  Run order was:
    1. EDA_MRI_complete.py
    2. EDA_Clinical.py
    3. Model1_Baseline_CNN.py
    4. Model2_TransferLearning.py
    5. Model3_Hybrid_Multimodal.py
    6. Fairness_Evaluation.py
    7. Explainability_SHAP_LIME_GradCAM.py  ← done
""")
